```
# @title Licensed under the Apache License, Version 2.0
# Vigil — Sistema de Análisis Electoral e Inteligencia Local
# Copyright 2026 RndmStudio
# Licensed under the Apache License, Version 2.0
```

# Vigil — Notebook 00: Setup y configuración del entorno

<a target="_blank" href="https://colab.research.google.com/github/TU_USUARIO/vigil/blob/main/notebooks/00_setup_and_config.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/>
</a>

## Overview

Este notebook inicializa el entorno del Sistema de Análisis Electoral e Inteligencia Local (Vigil).
Realiza las siguientes tareas:

1. Instala y verifica las dependencias del proyecto.
2. Carga la configuración desde `config/config.yaml` y variables de entorno (`.env`).
3. Valida las credenciales de Gemini API y Apify.
4. Verifica la estructura de carpetas del proyecto.
5. Ejecuta una prueba de conexión con `gemini-2.5-flash`.

**Ejecuta este notebook antes de cualquier otro en el proyecto.**

## Instalar dependencias

In [ ]:
%pip install -U -q 'google-genai>=1.0.0' polars apify-client networkx python-louvain python-dotenv pyyaml notebooklm-mcp-cli

## Cargar variables de entorno

En **local**: copia `.env.example` como `.env` y rellena tus credenciales.  
En **Colab**: agrega tus claves en *Secrets* (ícono de llave en el panel lateral).

In [ ]:
import os
import sys

# Detectar entorno: Colab o local
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    APIFY_API_TOKEN = userdata.get('APIFY_API_TOKEN')
else:
    from dotenv import load_dotenv
    load_dotenv('../.env')  # ruta relativa desde notebooks/
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')
    APIFY_API_TOKEN = os.environ.get('APIFY_API_TOKEN')

print(f"Entorno: {'Colab' if IN_COLAB else 'Local'}")
print(f"GEMINI_API_KEY: {'✓ configurada' if GEMINI_API_KEY else '✗ FALTA — revisa .env'}")
print(f"APIFY_API_TOKEN: {'✓ configurado' if APIFY_API_TOKEN else '✗ FALTA — revisa .env'}")

## Cargar configuración del proyecto

In [ ]:
import yaml
from pathlib import Path

# Ruta raíz del proyecto (un nivel arriba de notebooks/)
PROJECT_ROOT = Path('..').resolve()
CONFIG_PATH = PROJECT_ROOT / 'config' / 'config.yaml'

with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

print(f"Proyecto : {config['metadata']['nombre_proyecto']}")
print(f"Región   : {config['metadata']['region']}")
print(f"Periodo  : {config['metadata']['periodo_eleccion']}")
print(f"Versión  : {config['metadata']['version_vigil']}")
print(f"Candidatos: {[c['nombre'] for c in config['candidatos']]}")

## Verificar estructura del proyecto

In [ ]:
REQUIRED_DIRS = [
    'config', 'notebooks', 'src/ingestion', 'src/processing',
    'src/analysis', 'src/graph', 'data/raw', 'data/silver',
    'data/gold', 'docs', 'reports', 'evals', 'bin',
]

all_ok = True
for d in REQUIRED_DIRS:
    path = PROJECT_ROOT / d
    status = '✓' if path.exists() else '✗ FALTA'
    if not path.exists():
        all_ok = False
    print(f"  {status}  {d}")

print()
print('Estructura OK ✓' if all_ok else 'ADVERTENCIA: faltan carpetas — ejecuta setup en README')

## Seleccionar modelo Gemini

In [ ]:
MODEL_ID = "gemini-2.5-flash"  # @param ["gemini-2.5-flash", "gemini-2.5-pro", "gemini-2.0-flash"] {"allow-input": true, "isTemplate": true}

## Probar conexión con Gemini API

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=GEMINI_API_KEY)

response = client.models.generate_content(
    model=MODEL_ID,
    contents="Responde solo con: 'Vigil conectado correctamente.'",
    config=types.GenerateContentConfig(temperature=0.0),
)

print(response.text)

## Probar conexión con Apify

In [ ]:
from apify_client import ApifyClient

apify = ApifyClient(token=APIFY_API_TOKEN)

user_info = apify.user().get()
if user_info:
    plan = user_info.get('plan', {}).get('id', 'desconocido')
    print(f"Apify conectado ✓  |  Usuario: {user_info.get('username')}  |  Plan: {plan}")
else:
    print('Apify: no se pudo conectar — verifica APIFY_API_TOKEN')

## Resumen del entorno

Si todas las celdas anteriores muestran `✓`, el entorno está listo.  
El siguiente paso es **Notebook 01**: ingesta de datos desde Apify.

In [ ]:
import polars as pl
import networkx as nx

print("Versiones de librerías clave:")
print(f"  google-genai : {genai.__version__}")
print(f"  polars       : {pl.__version__}")
print(f"  networkx     : {nx.__version__}")
print()
print(f"Modelo activo  : {MODEL_ID}")
print(f"Región activa  : {config['metadata']['region']}")
print()
print("✓ Entorno Vigil (RndmStudio) inicializado correctamente.")